# Fine-Tuning BERT for POS Tagging & Chunking
### Data Science Internship – February 2026 | Assignment NLP-5

**Objective:** Fine-tune a transformer model (DistilBERT) to perform Part-of-Speech (POS) Tagging and Chunking (Phrase Detection) using token classification.

---

**Pipeline Flow:**
```
Raw Data → Tokenization → Label Alignment → Model Training → Evaluation → Inference → Comparison
```

**Model:** `distilbert-base-uncased`  
**Dataset:** CoNLL-2003 (POS Tags + Chunk Tags)  
**Tasks:** Token Classification — POS Tagging & Chunking

---
## Step 0: Install Required Libraries

In [ ]:
# Uncomment and run on Google Colab
# !pip install transformers datasets seqeval torch --quiet

---
## Step 1: Import Libraries

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
warnings.filterwarnings('ignore')

import torch
from torch.utils.data import Dataset

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import evaluate   # Hugging Face evaluate library (includes seqeval)

# Fix random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('All libraries imported successfully.')
print(f'PyTorch version : {torch.__version__}')
print(f'Device          : {device}')

---
## Task 1: Dataset Selection

**Dataset:** CoNLL-2003  
**Source:** Hugging Face Datasets Hub  

CoNLL-2003 is a standard benchmark dataset for sequence labeling tasks. It contains newswire text annotated with:
- **POS tags** — grammatical role of each word (noun, verb, adjective, etc.)
- **Chunk tags** — phrase-level groupings in IOB2 format (NP, VP, PP, etc.)
- **NER tags** — named entity labels (we do not use these in this assignment)

**Label types:**
- POS tags: NN, VBZ, DT, JJ, NNP, IN, PRP, RB, VBD, etc. (45 tags)
- Chunk tags: B-NP, I-NP, B-VP, I-VP, B-PP, O, etc. (23 tags)

In [ ]:
# Load CoNLL-2003 dataset
print('Loading CoNLL-2003 dataset...')
raw_datasets = load_dataset('conll2003', trust_remote_code=True)

print(f'Splits available : {list(raw_datasets.keys())}')
print(f'Train samples    : {len(raw_datasets["train"])}')
print(f'Val   samples    : {len(raw_datasets["validation"])}')
print(f'Test  samples    : {len(raw_datasets["test"])}')
print()

# Inspect a sample
sample = raw_datasets['train'][0]
print('Sample sentence:')
print(f'  Tokens     : {sample["tokens"]}')
print(f'  POS tags   : {sample["pos_tags"]}')
print(f'  Chunk tags : {sample["chunk_tags"]}')

In [ ]:
# Extract label lists from dataset features
pos_label_list   = raw_datasets['train'].features['pos_tags'].feature.names
chunk_label_list = raw_datasets['train'].features['chunk_tags'].feature.names

print(f'POS tag categories   ({len(pos_label_list)}) : {pos_label_list}')
print()
print(f'Chunk tag categories ({len(chunk_label_list)}) : {chunk_label_list}')

In [ ]:
# Visualise label distributions
def count_labels(dataset, tag_field):
    counts = {}
    for example in dataset:
        for tag in example[tag_field]:
            counts[tag] = counts.get(tag, 0) + 1
    return counts

pos_counts   = count_labels(raw_datasets['train'], 'pos_tags')
chunk_counts = count_labels(raw_datasets['train'], 'chunk_tags')

# Decode integer labels
pos_decoded   = {pos_label_list[k]: v   for k, v in pos_counts.items()}
chunk_decoded = {chunk_label_list[k]: v for k, v in chunk_counts.items()}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# POS distribution (top 15)
top_pos = dict(sorted(pos_decoded.items(), key=lambda x: x[1], reverse=True)[:15])
axes[0].bar(top_pos.keys(), top_pos.values(), color='#4299E1', edgecolor='white')
axes[0].set_title('Top 15 POS Tag Frequencies', fontweight='bold')
axes[0].set_xlabel('POS Tag')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Chunk distribution
top_chunk = dict(sorted(chunk_decoded.items(), key=lambda x: x[1], reverse=True))
axes[1].bar(top_chunk.keys(), top_chunk.values(), color='#48BB78', edgecolor='white')
axes[1].set_title('Chunk Tag Frequencies', fontweight='bold')
axes[1].set_xlabel('Chunk Tag')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---
## Task 2: Data Preprocessing — Tokenization & Label Alignment

BERT uses subword tokenization (WordPiece). A single word like "playing" may become ["play", "##ing"]. Since our labels are word-level, we must align them to the subword tokens.

**Strategy:**
- The first subword of a word gets the original label
- Subsequent subwords get label `-100` (ignored by PyTorch's CrossEntropyLoss)
- Special tokens `[CLS]` and `[SEP]` also get `-100`

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN    = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Tokenizer loaded: {MODEL_NAME}')

# Demonstrate subword alignment on a sample
demo_words  = ['John', 'works', 'at', 'headquarters']
demo_labels = [0, 1, 2, 3]  # dummy label IDs

encoding = tokenizer(
    demo_words,
    is_split_into_words=True,
    return_offsets_mapping=True
)

print('\nSubword tokenization demo:')
print(f'Words   : {demo_words}')
print(f'Subtokens: {tokenizer.convert_ids_to_tokens(encoding["input_ids"])}')
print(f'Word IDs : {encoding.word_ids()}')

In [ ]:
def tokenize_and_align_labels(examples, label_field):
    """
    Tokenize a batch of sentences and align token-level labels
    to the subword tokens produced by the BERT tokenizer.

    Args:
        examples    : batch of dataset examples
        label_field : 'pos_tags' or 'chunk_tags'

    Returns:
        tokenized batch with aligned 'labels' field
    """
    tokenized = tokenizer(
        examples['tokens'],
        is_split_into_words = True,   # input is already word-tokenized
        truncation          = True,
        max_length          = MAX_LEN,
        padding             = 'max_length'
    )

    all_labels = []

    for i, label_seq in enumerate(examples[label_field]):
        word_ids      = tokenized.word_ids(batch_index=i)
        aligned       = []
        previous_word = None

        for word_id in word_ids:
            if word_id is None:
                # Special tokens [CLS] and [SEP] — ignore in loss
                aligned.append(-100)
            elif word_id != previous_word:
                # First subword of a word — assign real label
                aligned.append(label_seq[word_id])
            else:
                # Subsequent subwords — ignore in loss
                aligned.append(-100)

            previous_word = word_id

        all_labels.append(aligned)

    tokenized['labels'] = all_labels
    return tokenized


print('Label alignment function defined.')

# Quick demo on a single sentence
demo_example = {
    'tokens'    : [['John', 'works', 'at', 'headquarters']],
    'pos_tags'  : [[22, 38, 11, 21]],  # NNP VBZ IN NN (dummy)
    'chunk_tags': [[3, 11, 4, 1]]
}
aligned = tokenize_and_align_labels(demo_example, 'pos_tags')
print(f'\nAligned labels example : {aligned["labels"][0][:15]}')
print('(-100 means subword/special token — ignored in loss calculation)')

In [ ]:
# Apply preprocessing to all splits for both POS and Chunk tasks
from functools import partial

print('Tokenizing and aligning labels for POS task...')
pos_tokenized = raw_datasets.map(
    partial(tokenize_and_align_labels, label_field='pos_tags'),
    batched=True,
    remove_columns=raw_datasets['train'].column_names
)

print('Tokenizing and aligning labels for Chunking task...')
chunk_tokenized = raw_datasets.map(
    partial(tokenize_and_align_labels, label_field='chunk_tags'),
    batched=True,
    remove_columns=raw_datasets['train'].column_names
)

print()
print('Preprocessing complete.')
print(f'POS   tokenized features : {list(pos_tokenized["train"].features.keys())}')
print(f'Chunk tokenized features : {list(chunk_tokenized["train"].features.keys())}')

---
## Task 3: Model Setup

In [ ]:
def build_model(label_list, model_name=MODEL_NAME):
    """
    Load a fresh AutoModelForTokenClassification with correct
    label count and id2label / label2id mappings.

    Args:
        label_list  : list of string label names
        model_name  : Hugging Face model identifier

    Returns:
        model, id2label, label2id
    """
    id2label   = {i: lbl for i, lbl in enumerate(label_list)}
    label2id   = {lbl: i for i, lbl in enumerate(label_list)}
    num_labels = len(label_list)

    model = AutoModelForTokenClassification.from_pretrained(
        model_name,
        num_labels = num_labels,
        id2label   = id2label,
        label2id   = label2id,
        ignore_mismatched_sizes = True
    )

    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Model           : {model_name}')
    print(f'Num labels      : {num_labels}')
    print(f'Total params    : {total:,}')
    print(f'Trainable params: {trainable:,}')

    return model, id2label, label2id


print('Model builder function defined.')

---
## Task 4 & 5: Training & Evaluation

We use the Hugging Face `Trainer` API with the `seqeval` metric for sequence-level evaluation.

In [ ]:
# Load seqeval metric
seqeval_metric = evaluate.load('seqeval')

def make_compute_metrics(id2label):
    """
    Returns a compute_metrics function that uses seqeval
    to calculate precision, recall, and F1 at sequence level.
    Ignores -100 labels (subwords / special tokens).
    """
    def compute_metrics(eval_preds):
        logits, labels = eval_preds
        predictions    = np.argmax(logits, axis=-1)

        true_preds, true_labels = [], []

        for pred_seq, label_seq in zip(predictions, labels):
            p_row, l_row = [], []
            for p, l in zip(pred_seq, label_seq):
                if l != -100:           # skip subword / special tokens
                    p_row.append(id2label[p])
                    l_row.append(id2label[l])
            true_preds.append(p_row)
            true_labels.append(l_row)

        results = seqeval_metric.compute(
            predictions=true_preds,
            references=true_labels
        )

        return {
            'precision' : results['overall_precision'],
            'recall'    : results['overall_recall'],
            'f1'        : results['overall_f1'],
            'accuracy'  : results['overall_accuracy'],
        }

    return compute_metrics


def get_training_args(output_dir, epochs=3, lr=2e-5, batch_size=16):
    """Return standard TrainingArguments for token classification."""
    return TrainingArguments(
        output_dir                  = output_dir,
        num_train_epochs            = epochs,
        learning_rate               = lr,
        per_device_train_batch_size = batch_size,
        per_device_eval_batch_size  = batch_size,
        weight_decay                = 0.01,
        evaluation_strategy         = 'epoch',
        save_strategy               = 'epoch',
        load_best_model_at_end      = True,
        logging_steps               = 50,
        seed                        = SEED,
        report_to                   = 'none'    # disable wandb
    )


print('Training utilities defined.')

### Model A: POS Tagging

In [ ]:
print('=' * 55)
print('MODEL A: POS TAGGING')
print('=' * 55)

# Build POS model
pos_model, pos_id2label, pos_label2id = build_model(pos_label_list)

# Data collator handles dynamic padding
data_collator = DataCollatorForTokenClassification(tokenizer)

# Training arguments
pos_args = get_training_args('./pos_model', epochs=3)

# Trainer
pos_trainer = Trainer(
    model           = pos_model,
    args            = pos_args,
    train_dataset   = pos_tokenized['train'],
    eval_dataset    = pos_tokenized['validation'],
    tokenizer       = tokenizer,
    data_collator   = data_collator,
    compute_metrics = make_compute_metrics(pos_id2label)
)

print('\nStarting POS Tagging training...')
pos_trainer.train()
print('POS Tagging training complete.')

In [ ]:
# Evaluate POS model on test set
print('Evaluating POS model on test set...')
pos_results = pos_trainer.evaluate(pos_tokenized['test'])

print('\nPOS TAGGING — Test Set Results:')
print(f'  Accuracy  : {pos_results["eval_accuracy"]:.4f}')
print(f'  Precision : {pos_results["eval_precision"]:.4f}')
print(f'  Recall    : {pos_results["eval_recall"]:.4f}')
print(f'  F1 Score  : {pos_results["eval_f1"]:.4f}')

### Model B: Chunking

In [ ]:
print('=' * 55)
print('MODEL B: CHUNKING')
print('=' * 55)

# Build Chunk model
chunk_model, chunk_id2label, chunk_label2id = build_model(chunk_label_list)

chunk_args = get_training_args('./chunk_model', epochs=3)

chunk_trainer = Trainer(
    model           = chunk_model,
    args            = chunk_args,
    train_dataset   = chunk_tokenized['train'],
    eval_dataset    = chunk_tokenized['validation'],
    tokenizer       = tokenizer,
    data_collator   = data_collator,
    compute_metrics = make_compute_metrics(chunk_id2label)
)

print('\nStarting Chunking training...')
chunk_trainer.train()
print('Chunking training complete.')

In [ ]:
# Evaluate Chunk model on test set
print('Evaluating Chunking model on test set...')
chunk_results = chunk_trainer.evaluate(chunk_tokenized['test'])

print('\nCHUNKING — Test Set Results:')
print(f'  Accuracy  : {chunk_results["eval_accuracy"]:.4f}')
print(f'  Precision : {chunk_results["eval_precision"]:.4f}')
print(f'  Recall    : {chunk_results["eval_recall"]:.4f}')
print(f'  F1 Score  : {chunk_results["eval_f1"]:.4f}')

---
## Task 6: Inference — Predict on Custom Sentences

In [ ]:
def predict_tags(sentence, model, tokenizer, id2label):
    """
    Predict token-level tags for a raw input sentence.

    Handles subword tokens by keeping only the prediction
    for the first subword of each original word.

    Args:
        sentence  : raw string sentence
        model     : fine-tuned token classification model
        tokenizer : corresponding tokenizer
        id2label  : integer-to-label mapping

    Returns:
        list of (word, predicted_tag) pairs
    """
    model.eval()

    # Tokenize preserving word boundaries
    words    = sentence.split()
    encoding = tokenizer(
        words,
        is_split_into_words = True,
        return_tensors      = 'pt',
        truncation          = True,
        max_length          = MAX_LEN
    ).to(device)

    word_ids = encoding.word_ids()

    with torch.no_grad():
        outputs = model(**encoding)

    logits      = outputs.logits
    predictions = torch.argmax(logits, dim=-1)[0].cpu().numpy()

    # Map predictions back to original words
    result        = []
    seen_word_ids = set()

    for token_idx, word_id in enumerate(word_ids):
        if word_id is None:
            continue
        if word_id not in seen_word_ids:
            result.append((words[word_id], id2label[predictions[token_idx]]))
            seen_word_ids.add(word_id)

    return result


def display_predictions(sentence, pos_preds, chunk_preds):
    """Print a formatted table of POS and Chunk predictions."""
    print(f'\nInput: "{sentence}"')
    print(f'{"Word":<18} {"POS Tag":<12} {"Chunk Tag"}')
    print('-' * 45)
    for (word, pos), (_, chunk) in zip(pos_preds, chunk_preds):
        print(f'{word:<18} {pos:<12} {chunk}')


# Move models to device
pos_model   = pos_model.to(device)
chunk_model = chunk_model.to(device)

# Test sentences
test_sentences = [
    'John works at Google in California',
    'The quick brown fox jumps over the lazy dog',
    'She bought a new laptop from the online store',
    'Artificial intelligence is transforming the world',
    'The president gave a speech at the White House',
]

print('=' * 55)
print('INFERENCE — CUSTOM SENTENCES')
print('=' * 55)

for sentence in test_sentences:
    pos_preds   = predict_tags(sentence, pos_model,   tokenizer, pos_id2label)
    chunk_preds = predict_tags(sentence, chunk_model, tokenizer, chunk_id2label)
    display_predictions(sentence, pos_preds, chunk_preds)
    print()

In [ ]:
# Visualise predictions for one sentence as a coloured table
def visualize_tags(sentence, pos_preds, chunk_preds):
    """Render POS and Chunk predictions as a colour-coded matplotlib table."""
    words  = [p[0] for p in pos_preds]
    pos    = [p[1] for p in pos_preds]
    chunks = [c[1] for c in chunk_preds]

    n = len(words)
    fig, ax = plt.subplots(figsize=(max(8, n * 1.5), 3))
    ax.axis('off')

    table_data   = [words, pos, chunks]
    row_labels   = ['Word', 'POS Tag', 'Chunk Tag']
    cell_colors  = []

    # Colour rows differently
    row_colours = ['#EBF8FF', '#E6FFFA', '#FFF5F5']
    for rc in row_colours:
        cell_colors.append([rc] * n)

    tbl = ax.table(
        cellText    = table_data,
        rowLabels   = row_labels,
        cellColours = cell_colors,
        loc         = 'center',
        cellLoc     = 'center'
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1, 2)

    ax.set_title(f'Predictions: "{sentence}"',
                 fontsize=11, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()


# Visualise for first sentence
s = test_sentences[0]
pp = predict_tags(s, pos_model,   tokenizer, pos_id2label)
cp = predict_tags(s, chunk_model, tokenizer, chunk_id2label)
visualize_tags(s, pp, cp)

---
## Task 7: Comparison — POS Tagging vs Chunking

In [ ]:
# Side-by-side metric comparison
comparison_data = {
    'Metric'   : ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'POS Tagging' : [
        round(pos_results['eval_accuracy'],  4),
        round(pos_results['eval_precision'], 4),
        round(pos_results['eval_recall'],    4),
        round(pos_results['eval_f1'],        4),
    ],
    'Chunking' : [
        round(chunk_results['eval_accuracy'],  4),
        round(chunk_results['eval_precision'], 4),
        round(chunk_results['eval_recall'],    4),
        round(chunk_results['eval_f1'],        4),
    ]
}

cdf = pd.DataFrame(comparison_data)
print('=' * 50)
print('POS TAGGING vs CHUNKING — METRIC COMPARISON')
print('=' * 50)
print(cdf.to_string(index=False))

# Bar chart
x     = np.arange(len(cdf['Metric']))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - width/2, cdf['POS Tagging'], width,
            label='POS Tagging', color='#4299E1', edgecolor='white')
b2 = ax.bar(x + width/2, cdf['Chunking'],    width,
            label='Chunking',    color='#48BB78', edgecolor='white')

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.005,
            f'{bar.get_height():.3f}',
            ha='center', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(cdf['Metric'])
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score')
ax.set_title('POS Tagging vs Chunking — DistilBERT Performance',
             fontweight='bold')
ax.legend()
ax.axhline(0.9, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Qualitative comparison table
qual = pd.DataFrame({
    'Dimension'       : [
        'Task level',
        'Output granularity',
        'Label set size',
        'Difficulty',
        'Context needed',
        'Use case'
    ],
    'POS Tagging' : [
        'Word-level grammar',
        'One tag per word',
        '45 tags (PTB)',
        'Easier',
        'Local (word + neighbours)',
        'Grammar analysis, parsing'
    ],
    'Chunking' : [
        'Phrase-level grouping',
        'IOB2 spans (B/I/O)',
        '23 tags',
        'Harder',
        'Wider span context',
        'IE, QA, NER pipelines'
    ]
})

print(qual.to_string(index=False))

---
## Task 8: Report — Observations & Insights

### POS Tagging vs Chunking — Key Differences

**POS Tagging** assigns a grammatical label to every word in a sentence. Each word receives exactly one tag drawn from a fixed tagset — noun, verb, adjective, preposition, and so on. The tags describe the syntactic role of each word in isolation, though context naturally influences the decision. For example, the word "bank" gets tagged differently in "river bank" versus "bank account" depending on the surrounding words.

**Chunking** (also called shallow parsing) goes one level higher. Instead of labelling individual words, it groups consecutive words into phrases — noun phrases (NP), verb phrases (VP), prepositional phrases (PP), and so on. It uses the IOB2 labelling scheme: B- marks the beginning of a chunk, I- marks continuation inside a chunk, and O marks words outside any chunk.

### Challenges Faced

1. **Subword alignment** is the most critical preprocessing challenge. BERT's WordPiece tokenizer splits words like "running" into ["running"] or uncommon words into multiple pieces. Labels are word-level, so you must carefully map each subword back to its original word and assign -100 to non-first subwords to exclude them from the loss calculation.

2. **IOB2 consistency in chunking.** The model must learn that a B- tag can only be followed by an I- tag of the same type or an O/B- tag of a different type. Violations of this constraint can appear in raw model outputs and need post-processing for production systems.

3. **Label imbalance.** In POS tagging, noun (NN) and verb (VBZ) tags dominate. In chunking, noun phrases (B-NP, I-NP) make up the majority of non-O tags. The model naturally performs better on frequent classes.

### Observations

- DistilBERT achieves strong results on both tasks despite being 40% smaller than BERT. The distillation process retains most of the linguistic knowledge needed for token classification.
- POS tagging tends to converge faster and achieve higher raw accuracy because the label boundaries are cleaner — each word has exactly one unambiguous tag.
- Chunking is inherently harder because the model must learn both the content of a phrase (what type it is) and its boundaries (where it starts and ends). F1 on chunking is a stricter metric — a span is only counted as correct if both its type and exact boundaries match.
- The `seqeval` metric is the right choice for both tasks because it evaluates at the span level rather than the token level, which gives a more honest picture of chunking performance.